# 01. Data understanding and preprocessing
HW2: Multiple Object Video Detection (UCU AI 2026)

The organisers already provide COCO annotations, YOLO labels and a train/val split by video, so no format conversion is needed.
This notebook (CPU only) does four things:
1. **EDA** on the COCO annotations: class balance, object sizes, scene density, differences between videos, motion between frames. Conclusions go into `experiments_log.md`.
2. **Sanity checks**: test image size vs `test_images.csv`, boxes drawn on frames, YOLO labels vs COCO.
3. **Frame subsampling** (every N-th frame of every video) and **masking of ignored regions** in the training images.
4. Saves the result in `OUT`, to be stored as a Kaggle Dataset and used by the next notebooks.

## 0. Config
Change only this cell.

In [ ]:
from pathlib import Path
import json, random, shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm.auto import tqdm

ROOT = Path("/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026")
OUT = Path("/kaggle/working/hw2_sub")   # result: subsampled frames + labels + COCO json of the subsets

STEP_TRAIN = 4        # keep every 4th frame of every training video
STEP_VAL = 4          # same for validation (faster evaluation)
MASK_IGNORED = True   # paint ignored regions grey in TRAIN images (val and test stay untouched)
SEED = 42
random.seed(SEED); np.random.seed(SEED)

## 1. What is in the data folder

In [ ]:
print("root :", sorted(p.name for p in ROOT.iterdir()))
for s in ["train", "val", "test"]:
    print(f"{s:5s}:", sorted(p.name for p in (ROOT / s).iterdir()))
print()
print((ROOT / "train" / "data.yaml").read_text())

In [ ]:
# Load COCO annotations of train and val into DataFrames
DATA = {}
for split in ["train", "val"]:
    raw = json.loads((ROOT / split / "annotations_coco.json").read_text())
    imgs = pd.DataFrame(raw["images"])
    anns = pd.DataFrame(raw["annotations"])
    bb = np.array(anns["bbox"].tolist(), dtype=float)
    anns = anns.drop(columns="bbox").assign(x=bb[:, 0], y=bb[:, 1], w=bb[:, 2], h=bb[:, 3])
    DATA[split] = {"imgs": imgs, "anns": anns}
    if split == "train":
        META = {"info": raw.get("info", {}), "categories": raw["categories"]}
    del raw

CATS = {c["id"]: c["name"] for c in META["categories"]}
print("classes:", CATS)

# the provided split must be disjoint by video (no leakage between train and val)
assert not set(DATA["train"]["imgs"].video) & set(DATA["val"]["imgs"].video), "train and val share videos!"
print("train and val are disjoint by video")

def objects(split):
    """Real objects only (iscrowd == 0). iscrowd == 1 marks ignored regions."""
    a = DATA[split]["anns"]
    return a[a.iscrowd == 0]

rows = []
for s, d in DATA.items():
    a, im = d["anns"], d["imgs"]
    cr = a[a.iscrowd == 1]
    rows.append({"split": s, "videos": im.video_id.nunique(), "frames": len(im),
                 "objects": int((a.iscrowd == 0).sum()), "crowd_annotations": len(cr),
                 "unique_ignored_regions": len(cr[["image_id", "x", "y", "w", "h"]].drop_duplicates())})
print(pd.DataFrame(rows).to_string(index=False))
print("\nimage sizes (width, height):")
for s, d in DATA.items():
    print(" ", s, dict(d["imgs"][["width", "height"]].value_counts().items()))
print("\nHow original VisDrone classes were merged (rows: orig_category_id, columns: category_id):")
print(DATA["train"]["anns"].groupby(["orig_category_id", "category_id"]).size().unstack(fill_value=0).to_string())

## 2. EDA
Goal: facts that justify later choices (input resolution, anchors, number of queries, `max_det`, frame step, augmentation).

### 2.1 Class balance (train vs val)

In [ ]:
share = pd.DataFrame({s: objects(s).category_id.map(CATS).value_counts(normalize=True) for s in DATA}).round(4)
counts = objects("train").category_id.map(CATS).value_counts()
print(pd.concat([counts.rename("train_objects"), share.add_suffix("_share")], axis=1).to_string())

ax = share.plot.bar(figsize=(8, 3.5), title="Share of objects per class")
ax.set_yscale("log"); plt.tight_layout(); plt.show()

### 2.2 Object sizes
COCO buckets by box area: small < 32², medium < 96², large otherwise. The last block shows how many objects become tiny after resizing the frame to the network input.

In [ ]:
BINS, LABELS = [0, 32 ** 2, 96 ** 2, np.inf], ["small", "medium", "large"]
sizes = {}
for s in DATA:
    o = objects(s).copy()
    o["sz"] = np.sqrt(o.w * o.h)
    o["bucket"] = pd.cut(o.w * o.h, BINS, labels=LABELS, right=False)
    sizes[s] = o

print("Share by COCO bucket:")
print(pd.DataFrame({s: sizes[s].bucket.value_counts(normalize=True) for s in sizes}).round(3).to_string())
print("\nShare by bucket, per class (train):")
print(sizes["train"].groupby(sizes["train"].category_id.map(CATS)).bucket.value_counts(normalize=True).unstack().round(2).to_string())

o = sizes["train"]
print("\nsqrt(area) in px, percentiles 10/50/90:", o.sz.quantile([.1, .5, .9]).round(1).tolist(),
      "| median box %.0f x %.0f" % (o.w.median(), o.h.median()))

W0 = int(DATA["train"]["imgs"].width.mode()[0])
print(f"\nShare of objects with sqrt(area) below 8 px after resizing the frame width {W0} -> imgsz:")
for imgsz in (640, 960, 1280, 1344):
    print(f"  imgsz {imgsz}: {(o.sz * imgsz / W0 < 8).mean():.1%}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for s in sizes:
    ax[0].hist(sizes[s].sz.clip(upper=200), bins=80, alpha=.5, density=True, label=s)
    ax[1].hist((sizes[s].h / sizes[s].w.clip(lower=1)).clip(upper=5), bins=80, alpha=.5, density=True, label=s)
ax[0].set_title("Box size, sqrt(area) px (clipped at 200)"); ax[1].set_title("Aspect ratio h/w"); ax[0].legend()
plt.tight_layout(); plt.show()

### 2.3 Scene density and differences between videos

In [ ]:
def frame_counts(split):
    d = DATA[split]
    return objects(split).groupby("image_id").size().reindex(d["imgs"].id, fill_value=0)

fc = {s: frame_counts(s) for s in DATA}
for s, c in fc.items():
    print(f"{s}: objects per frame mean={c.mean():.1f} median={c.median():.0f} p95={c.quantile(.95):.0f} "
          f"p99={c.quantile(.99):.0f} max={c.max()} | empty frames: {(c == 0).mean():.1%}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for s, c in fc.items():
    ax[0].hist(c.clip(upper=300), bins=60, alpha=.5, density=True, label=s)
ax[0].set_title("Objects per frame (clipped at 300)"); ax[0].legend()

d = DATA["train"]
vid = d["imgs"].set_index("id").video
per_video = fc["train"].groupby(vid.reindex(fc["train"].index).values).agg(frames="count", obj_per_frame="mean", max_objs="max").round(1)
per_video.obj_per_frame.plot.hist(bins=25, ax=ax[1], title="Mean objects per frame, per train video")
plt.tight_layout(); plt.show()
print("\nSparsest and densest train videos:")
print(per_video.sort_values("obj_per_frame").iloc[[0, 1, 2, -3, -2, -1]].to_string())
print("\nFrames per train video: min=%d median=%d max=%d" % (per_video.frames.min(), per_video.frames.median(), per_video.frames.max()))

### 2.4 Ignored regions

In [ ]:
for s, d in DATA.items():
    a, im = d["anns"], d["imgs"]
    cr = a[a.iscrowd == 1].drop_duplicates(["image_id", "x", "y", "w", "h"])
    n_reg = cr.groupby("image_id").size().reindex(im.id, fill_value=0)
    frame_area = im.set_index("id").eval("width * height")
    cover = ((cr.w * cr.h).groupby(cr.image_id).sum().reindex(im.id, fill_value=0) / frame_area).clip(upper=1)
    print(f"{s}: frames with ignored regions {(n_reg > 0).mean():.1%} | regions per frame (mean) {n_reg.mean():.1f} "
          f"| share of frame area covered (approx., mean) {cover.mean():.1%}")

### 2.5 Frame step (justification for subsampling)
Consecutive frames of one video are near-duplicates. For every tracked object, measure how far it moves between frame t and t+k, in units of its own size. A small shift means that skipping frames loses little information.

In [ ]:
im = DATA["train"]["imgs"].rename(columns={"id": "image_id"})[["image_id", "video_id", "frame_id"]]
o = objects("train").merge(im, on="image_id")
o["cx"], o["cy"], o["s"] = o.x + o.w / 2, o.y + o.h / 2, np.sqrt(o.w * o.h)
base = o[["video_id", "track_id", "frame_id", "cx", "cy", "s"]]
shift = {}
for k in (1, 2, 4, 8, 16):
    nxt = base[["video_id", "track_id", "frame_id", "cx", "cy"]].copy()
    nxt["frame_id"] -= k
    m = base.merge(nxt, on=["video_id", "track_id", "frame_id"], suffixes=("", "_n"))
    shift[k] = (np.hypot(m.cx_n - m.cx, m.cy_n - m.cy) / m.s).median()
print("Median shift of an object between frames t and t+k, in box sizes:")
print(pd.Series(shift).round(3).rename_axis("k").to_string())
del o, base

## 3. Sanity checks
### 3.1 Test frames vs `test_images.csv`
Train/val frames are 1344x756 but the CSV lists 1920x1080 for test. Check what the test JPEGs really are: this decides in which coordinates the submission boxes must be.

In [ ]:
test_df = pd.read_csv(ROOT / "test_images.csv")
print(len(test_df), "test frames,", test_df["clip"].nunique(), "clips")
print("sizes listed in the CSV:", dict(test_df[["width", "height"]].value_counts().items()))
print("\nactual JPEG sizes (random sample of 8):")
bad = 0
for r in test_df.sample(8, random_state=0).itertuples():
    w, h = Image.open(ROOT / r.file_name).size
    bad += (w, h) != (r.width, r.height)
    print(" ", r.file_name, (w, h), "| csv:", (r.width, r.height))
print("\nmismatches in the sample:", bad)

### 3.2 Ground truth drawn on frames
Coloured = objects (one colour per class), red dashed = ignored regions. Boxes must sit on real objects, in several different videos.

In [ ]:
cmap = plt.get_cmap("tab10")

def show_gt(split, n=6, seed=0):
    d = DATA[split]
    sel = d["imgs"].sample(n, random_state=seed)
    a = d["anns"][d["anns"].image_id.isin(sel.id)]
    cols = 3
    fig, axes = plt.subplots((n + cols - 1) // cols, cols, figsize=(6 * cols, 3.6 * ((n + cols - 1) // cols)))
    for ax, r in zip(np.atleast_1d(axes).flatten(), sel.itertuples()):
        ax.imshow(Image.open(ROOT / split / r.file_name))
        g = a[a.image_id == r.id]
        for q in g[g.iscrowd == 1].drop_duplicates(["x", "y", "w", "h"]).itertuples():
            ax.add_patch(patches.Rectangle((q.x, q.y), q.w, q.h, fill=False, ec="red", lw=2, ls="--"))
        for q in g[g.iscrowd == 0].itertuples():
            ax.add_patch(patches.Rectangle((q.x, q.y), q.w, q.h, fill=False, ec=cmap((q.category_id - 1) % 10), lw=1))
        ax.set_title(f"{r.video} / {r.frame_id}  ({(g.iscrowd == 0).sum()} objects)", fontsize=8); ax.axis("off")
    plt.tight_layout(); plt.show()

show_gt("train")

In [ ]:
show_gt("val", seed=1)

### 3.3 YOLO labels vs COCO annotations
The training labels (`labels/`) must agree with the COCO file (same objects, class ids shifted by one).

In [ ]:
def yolo_label_path(split, file_name):
    return ROOT / split / "labels" / Path(file_name).relative_to("images").with_suffix(".txt")

def check_labels(split, n=200, seed=0):
    d = DATA[split]
    sel = d["imgs"].sample(min(n, len(d["imgs"])), random_state=seed)
    o = objects(split)
    o = o[o.image_id.isin(sel.id)]
    n_coco = o.groupby("image_id").size().reindex(sel.id, fill_value=0).values
    n_yolo, cls_yolo = [], []
    for r in sel.itertuples():
        p = yolo_label_path(split, r.file_name)
        lines = [l.split() for l in p.read_text().splitlines() if l.strip()] if p.exists() else []
        n_yolo.append(len(lines)); cls_yolo += [int(l[0]) for l in lines]
    n_yolo = np.array(n_yolo)
    print(f"{split}: frames with identical object count {(n_coco == n_yolo).mean():.1%} | total COCO {n_coco.sum()} vs YOLO {n_yolo.sum()}")
    h_y = np.bincount(cls_yolo, minlength=len(CATS)) / max(len(cls_yolo), 1)
    h_c = np.bincount(o.category_id.values - 1, minlength=len(CATS)) / max(len(o), 1)
    print("  class shares YOLO:", h_y.round(3).tolist())
    print("  class shares COCO:", h_c.round(3).tolist())

check_labels("train"); check_labels("val")

## 4. Frame subsampling and ignored regions
Decisions (explain them in the report):
- **Split**: provided by the organisers and made of whole videos (no frame-level leakage), checked above.
- **Step**: consecutive frames are near-duplicates (section 2.5), so every N-th frame keeps most of the information and cuts the epoch time N times. `STEP_TRAIN` / `STEP_VAL` set N.
- **Ignored regions**: objects inside them are not annotated. Left as they are, they would teach the detector that real objects are background. In the train images they are painted grey, in the val images they stay untouched and the COCO evaluator (`iscrowd=1`) skips detections inside them.
- Frames, YOLO labels and COCO json of both subsets (`train_sub.json`, `val_sub.json`) are written to `OUT` with the same relative paths as the original data.

In [ ]:
def select_frames(imgs, step):
    """Every step-th frame of every video (sorted by frame number)."""
    imgs = imgs.sort_values(["video_id", "frame_id"]).copy()
    imgs["rank"] = imgs.groupby("video_id").cumcount()
    return imgs[imgs["rank"] % step == 0]

def build_subset(split, step, mask):
    d = DATA[split]
    sel = select_frames(d["imgs"], step)
    crowd = d["anns"][(d["anns"].iscrowd == 1) & d["anns"].image_id.isin(sel.id)]
    regions = {k: g[["x", "y", "w", "h"]].values
               for k, g in crowd.drop_duplicates(["image_id", "x", "y", "w", "h"]).groupby("image_id")}
    for r in tqdm(list(sel.itertuples()), desc=split):
        src, dst = ROOT / split / r.file_name, OUT / split / r.file_name
        dst.parent.mkdir(parents=True, exist_ok=True)
        if mask and r.id in regions:
            img = Image.open(src).convert("RGB")
            for x, y, w, h in regions[r.id]:
                img.paste((114, 114, 114), (int(x), int(y), int(x + w), int(y + h)))
            img.save(dst, quality=95)
        else:
            shutil.copy2(src, dst)
        lab_src = yolo_label_path(split, r.file_name)
        lab_dst = OUT / split / "labels" / Path(r.file_name).relative_to("images").with_suffix(".txt")
        lab_dst.parent.mkdir(parents=True, exist_ok=True)
        if lab_src.exists():
            shutil.copy2(lab_src, lab_dst)
        else:
            lab_dst.write_text("")          # frame without objects
    return sel

OUT.mkdir(parents=True, exist_ok=True)
sel_train = build_subset("train", STEP_TRAIN, MASK_IGNORED)
sel_val = build_subset("val", STEP_VAL, False)
print(len(sel_train), "train frames,", len(sel_val), "val frames")

In [ ]:
# COCO json of both subsets (ground truth for the evaluation with pycocotools, annotations for torchvision models)
def subset_coco(split, sel):
    a = DATA[split]["anns"]
    a = a[a.image_id.isin(sel.id)]
    return {
        "info": META["info"],
        "categories": META["categories"],
        "images": [{"id": int(r.id), "file_name": r.file_name, "width": int(r.width), "height": int(r.height),
                    "video_id": int(r.video_id), "video": r.video, "frame_id": int(r.frame_id)} for r in sel.itertuples()],
        "annotations": [{"id": int(r.id), "image_id": int(r.image_id), "category_id": int(r.category_id),
                         "bbox": [r.x, r.y, r.w, r.h],
                         "area": float(r.area) if pd.notna(r.area) else float(r.w * r.h),
                         "iscrowd": int(r.iscrowd),
                         "track_id": int(r.track_id) if pd.notna(r.track_id) else -1} for r in a.itertuples()],
    }

for split, sel in (("train", sel_train), ("val", sel_val)):
    js = subset_coco(split, sel)
    (OUT / f"{split}_sub.json").write_text(json.dumps(js))
    n_obj = sum(a["iscrowd"] == 0 for a in js["annotations"])
    print(f"{split}_sub.json: {len(js['images'])} frames, {n_obj} objects")
(OUT / "subset_info.json").write_text(json.dumps({"step_train": STEP_TRAIN, "step_val": STEP_VAL, "mask_ignored": MASK_IGNORED,
                                                  "n_train": len(sel_train), "n_val": len(sel_val), "seed": SEED}))

### 4.1 Check what the model will actually see
Train frames from `OUT` with YOLO labels drawn on top. Grey patches are masked ignored regions: no object should be left half-covered, and the labels must sit on the visible objects.

In [ ]:
files = sorted((OUT / "train" / "images").rglob("*.jpg"))
pick = random.Random(1).sample(files, min(6, len(files)))
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, f in zip(axes.flatten(), pick):
    img = Image.open(f); W, H = img.size
    ax.imshow(img)
    lab = OUT / "train" / "labels" / f.relative_to(OUT / "train" / "images").with_suffix(".txt")
    for l in lab.read_text().splitlines():
        c, cx, cy, w, h = l.split(); c = int(c); cx, cy, w, h = (float(v) for v in (cx, cy, w, h))
        ax.add_patch(patches.Rectangle(((cx - w / 2) * W, (cy - h / 2) * H), w * W, h * H, fill=False, ec=cmap(c % 10), lw=1))
    ax.set_title(f"{f.parent.name} / {f.stem}", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()
print("size of OUT: %.2f GB" % (sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file()) / 1e9))

## Next
1. Write the EDA conclusions and the decisions above into `experiments_log.md`: rare classes, share of small objects and what it means for the input size, density (`max_det`, number of queries), how different the videos are, the largest frame step with a small shift, coverage of ignored regions, test size vs train size.
2. *Save Version* (Save & Run All), then in the Output tab choose *New Dataset* (e.g. `hw2-sub`).
3. Continue in `02_yolo_baseline.ipynb`.